# Fashion-MNIST — 5 классов (MLP, TensorFlow/Keras)

Оставляем только: T-shirt/top, Trouser, Pullover, Dress, Sneaker.

## 1. Импорты и константы

In [ ]:
import os
import ssl
import time

import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

FASHION_MNIST_MEAN, FASHION_MNIST_STD = 0.2860, 0.3530

# Классы, которые нужно оставить (оригинальные индексы Fashion-MNIST)
KEEP_CLASSES = [0, 1, 2, 3, 7]  # T-shirt/top, Trouser, Pullover, Dress, Sneaker
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Sneaker"]

# Старый индекс -> новый индекс (0..4)
LABEL_MAP = {old: new for new, old in enumerate(KEEP_CLASSES)}

print("using device:", "GPU" if tf.config.list_physical_devices("GPU") else "CPU")

## 2. Загрузка и фильтрация данных до 5 классов

Метки переиндексируются в диапазон 0..4, чтобы их можно было использовать напрямую в `SparseCategoricalCrossentropy` на 5 выходов.

In [4]:
def filter_and_remap(x, y):
    keep_mask = np.isin(y, KEEP_CLASSES)
    x_filtered = x[keep_mask]
    y_filtered = y[keep_mask]
    y_remapped = np.vectorize(LABEL_MAP.get)(y_filtered)
    return x_filtered, y_remapped


def load_fashion_mnist_5classes():
    (train_x, train_y), (test_x, test_y) = keras.datasets.fashion_mnist.load_data()

    train_x = (train_x.astype("float32") / 255.0 - FASHION_MNIST_MEAN) / FASHION_MNIST_STD
    test_x = (test_x.astype("float32") / 255.0 - FASHION_MNIST_MEAN) / FASHION_MNIST_STD

    train_x, train_y = filter_and_remap(train_x, train_y)
    test_x, test_y = filter_and_remap(test_x, test_y)
    return (train_x, train_y), (test_x, test_y)

## 3. Данные

In [3]:
(train_x, train_y), (test_x, test_y) = load_fashion_mnist_5classes()
print(f"train: {len(train_x)} примеров, test: {len(test_x)} примеров")
print("классы:", dict(enumerate(CLASS_NAMES)))

train: 30000 примеров, test: 5000 примеров
классы: {0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Sneaker'}


## 4. Модель

MLP: два скрытых слоя с BatchNorm + Dropout, финальный слой на 5 классов.

In [5]:
model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(len(KEEP_CLASSES)),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

## 5. Колбэк для замера времени эпох

In [6]:
class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.total_start = time.time()

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        print(f"epoch {epoch + 1} took {time.time() - self.epoch_start:.2f}s")

    def on_train_end(self, logs=None):
        print(f"total training time: {time.time() - self.total_start:.2f}s")

## 6. Обучение

In [7]:
model.fit(
    train_x, train_y,
    validation_data=(test_x, test_y),
    batch_size=128,
    epochs=10,
    callbacks=[EpochTimer()],
)

Epoch 1/10
213/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9290 - loss: 0.2231epoch 1 took 1.26s
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9302 - loss: 0.2178 - val_accuracy: 0.9424 - val_loss: 0.1688
Epoch 2/10
215/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9533 - loss: 0.1425epoch 2 took 0.44s
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9534 - loss: 0.1434 - val_accuracy: 0.9502 - val_loss: 0.1594
Epoch 3/10
217/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9606 - loss: 0.1224epoch 3 took 0.50s
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9604 - loss: 0.1232 - val_accuracy: 0.9558 - val_loss: 0.1378
Epoch 4/10
223/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9631 - loss: 0.1082epoch 4 took 0.43s
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9632 - loss: 0.1080 - val_accuracy: 0.9566 - val_loss: 0.1340
Epoch 5/10
221/235 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9657 - loss: 0.1036epoch 5 took 0.43s
235/235 ━━━━

In [ ]:
test_loss, test_acc = model.evaluate(test_x, test_y, verbose=0)
print(f"test accuracy: {test_acc:.4f}")

## 7. Сохранение модели

In [ ]:
model.save("fashion_mnist_5classes_mlp_tf.keras")
print("model saved to fashion_mnist_5classes_mlp_tf.keras")

## 10. Transfer learning на MNIST (цифры, 10 классов)

Замораживаем все слои, кроме последнего, заменяем выходной слой на 10 классов и дообучаем только его на MNIST.

### 10.1 Данные MNIST

In [ ]:
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

(mnist_train_x, mnist_train_y), (mnist_test_x, mnist_test_y) = keras.datasets.mnist.load_data()

mnist_train_x = (mnist_train_x.astype("float32") / 255.0 - MNIST_MEAN) / MNIST_STD
mnist_test_x = (mnist_test_x.astype("float32") / 255.0 - MNIST_MEAN) / MNIST_STD

print(f"mnist train: {len(mnist_train_x)} примеров, test: {len(mnist_test_x)} примеров")

### 10.2 Заморозка слоёв и замена выходного слоя

Берём все слои обученной модели, кроме последнего `Dense(5)`, замораживаем их (`trainable = False`) и достраиваем новый выходной слой на 10 классов.

In [ ]:
NUM_DIGIT_CLASSES = 10

# все слои, кроме последнего Dense(5) — тот отбрасываем
base_layers = model.layers[:-1]

for layer in base_layers:
    layer.trainable = False

transfer_model = keras.Sequential(
    base_layers + [layers.Dense(NUM_DIGIT_CLASSES, name="digits_head")]
)
transfer_model.build(input_shape=(None, 28, 28))

transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

transfer_model.summary()

### 10.3 Дообучение выходного слоя на MNIST

In [ ]:
transfer_model.fit(
    mnist_train_x, mnist_train_y,
    validation_data=(mnist_test_x, mnist_test_y),
    batch_size=128,
    epochs=10,
    callbacks=[EpochTimer()],
)

In [ ]:
mnist_test_loss, mnist_test_acc = transfer_model.evaluate(mnist_test_x, mnist_test_y, verbose=0)
print(f"test accuracy: {mnist_test_acc:.4f}")

### 10.4 Сохранение дообученной модели

In [ ]:
transfer_model.save("mnist_digits_transfer_tf.keras")
print("model saved to mnist_digits_transfer_tf.keras")

## 11. Fine-tuning: если точности transfer learning не хватает

76% на MNIST — это низко, потому что признаки из `fc1`/`fc2` заточены под силуэты одежды, а не под штрихи цифр, и учится только последний линейный слой (1290 параметров) поверх них.

Стандартный следующий шаг — **разморозить часть базовой сети и дообучить её с маленьким learning rate**. Порядок действий:

1. Сначала обучить только новую голову (уже сделано в разделе 10) — это "прогревает" веса головы, не давая случайным градиентам испортить уже обученные признаки.
2. Разморозить последние 1-2 слоя базовой сети (обычно ближе к выходу — они более специфичны для исходной задачи, тогда как ранние слои содержат более общие признаки).
3. Пересобрать (`compile`) модель с **меньшим learning rate** (обычно в 10-100 раз меньше исходного) — иначе большие градиенты быстро "сотрут" полезные предобученные веса.
4. Дообучить (`fit`) ещё несколько эпох.

Ниже — размораживаем `fc2`/`bn2` (второй скрытый слой) и дообучаем всю модель, кроме первого слоя `fc1`/`bn1`.

In [ ]:
# размораживаем второй скрытый слой (dense_1 + batch_normalization_1),
# первый слой (dense, batch_normalization) и Flatten/Dropout/Activation оставляем как есть
FINE_TUNE_LAYER_NAMES = {"dense_1", "batch_normalization_1"}

for layer in transfer_model.layers:
    if layer.name in FINE_TUNE_LAYER_NAMES:
        layer.trainable = True

# меньший learning rate, чтобы не разрушить предобученные веса
transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

transfer_model.summary()

In [ ]:
transfer_model.fit(
    mnist_train_x, mnist_train_y,
    validation_data=(mnist_test_x, mnist_test_y),
    batch_size=128,
    epochs=10,
    callbacks=[EpochTimer()],
)

fine_tuned_loss, fine_tuned_acc = transfer_model.evaluate(mnist_test_x, mnist_test_y, verbose=0)
print(f"test accuracy after fine-tuning: {fine_tuned_acc:.4f}")

In [ ]:
transfer_model.save("mnist_digits_finetuned_tf.keras")
print("model saved to mnist_digits_finetuned_tf.keras")